In [ ]:
from IPython.display import display
import os
from pathlib import Path
import bacpipe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap

In [23]:
import importlib.resources as pkg_resources
os.chdir(pkg_resources.files("bacpipe"))
os.chdir('..')

MODEL_NAME = 'aves_especies'                  # name of the model to run. Supported models are in bacpipe.supported_models
MARSH_DIR = Path("bacpipe/data_full/audio/")
SWAMP_DIR = Path("bacpipe/data_full/swamp")
# AUDIO_DIR = Path("bacpipe/data")
# Must match the results directory used when training the probe.
RESULTS_DIR = bacpipe.settings.main_results_dir

In [24]:
# Generate or load embeddings for annotated marsh recordings
marsh_loader = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,
    audio_dir=str(MARSH_DIR),
)
marsh_embeds = marsh_loader.embeddings(return_type="array")

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

aves_especies checkpoint exists.




###### Generating embeddings using AVES_ESPECIES ######

finding audio files: 113it [00:00, 1569.48it/s]
Found 104 number of audio files.
Using device='cuda'
                                                                  

In [26]:
swamp_loader = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,
    audio_dir=str(SWAMP_DIR),
)
swamp_embeds = swamp_loader.embeddings(return_type="array")

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.



aves_especies checkpoint exists.




###### Generating embeddings using AVES_ESPECIES ######

finding audio files: 280it [00:00, 19036.28it/s]
Found 277 number of audio files.
Using device='cuda'
                                                                  

In [27]:
print("MARSH_DIR:", Path(MARSH_DIR).resolve())

print("Marsh annotations exist:",
      (Path(MARSH_DIR) / "annotations.csv").is_file())

print("Marsh audio files:",
      len(list(Path(MARSH_DIR).glob("*.wav"))))

print("Marsh annotation preview:")
import pandas as pd

annotations = pd.read_csv(Path(MARSH_DIR) / "annotations.csv")
print(annotations.shape)
print(annotations.head())
print("Unique annotation filenames:",
      annotations["audiofilename"].nunique())

MARSH_DIR: /workspaces/OsaAnurans3.11/bacpipe/data_full/audio
Marsh annotations exist: True
Marsh audio files: 0
Marsh annotation preview:
(697, 4)
                          audiofilename      start        end label:species
0  Colombia/H23/H23_20230423_183500.WAV   0.376344  59.784925        DENMIC
1  Colombia/H23/H23_20230424_172000.WAV   7.043008   7.486557        DENEBR
2  Colombia/H23/H23_20230424_172000.WAV  20.483068  21.450810        DENMIC
3  Colombia/H23/H23_20230424_172000.WAV  33.913237  34.464312        DENEBR
4  Colombia/H23/H23_20230424_172000.WAV  40.247524  41.000212        DENEBR
Unique annotation filenames: 104


In [28]:
import bacpipe

print("Bacpipe module:", bacpipe.__file__)
print("Evaluation task:", bacpipe.config.evaluation_task)
print("Results directory:", bacpipe.settings.main_results_dir)

Bacpipe module: /workspaces/OsaAnurans3.11/bacpipe/__init__.py
Evaluation task: ['probing']
Results directory: bacpipe_results


In [29]:
import inspect
import bacpipe

print("Bacpipe:", bacpipe.__file__, flush=True)
print("Evaluation function:", inspect.getfile(bacpipe.model_specific_evaluation), flush=True)

Bacpipe: /workspaces/OsaAnurans3.11/bacpipe/__init__.py
Evaluation function: /workspaces/OsaAnurans3.11/bacpipe/core/workflows.py


In [30]:
# def reshape_audiofilename(path, feature_name = "audiofilename"):
#     curr_df = pd.read_csv(path)
#     curr_df[feature_name] = curr_df[feature_name].apply(lambda x: "audio/colombia/" + x)
#     curr_df.to_csv(path, index=None)
MARSH_DIR

PosixPath('bacpipe/data_full/audio')

In [32]:
# Align marsh annotations with the marsh embedding windows
marsh_gt = bacpipe.ground_truth_by_model(
    model=MODEL_NAME,
    audio_dir=str(MARSH_DIR),
    annotations_filename="annotations.csv",
    overwrite=True,
)


Multiple embeddings found for model aves_especies in bacpipe_results/audio/embeddings. Using the most recent path.

                                                                                                              
The simultaneous labels column of the ground truth has values exceeding 1. This means you have multi-label ground truth annotations. If this should not be happening ensure the ground truth is created correcly.



In [37]:

print("Embedding shape:", marsh_embeds.shape)
print("Ground truth shape:", marsh_gt.shape)
print("Ground truth type:", type(marsh_gt))

Embedding shape: (6240, 768)
Ground truth shape: (6240, 9)
Ground truth type: <class 'pandas.core.frame.DataFrame'>


In [38]:
if hasattr(marsh_gt, "columns"):
    print("Ground truth columns:", marsh_gt.columns.tolist())
    print(marsh_gt.head())
elif isinstance(marsh_gt, dict):
    print("Ground truth keys:", marsh_gt.keys())
    for key, value in marsh_gt.items():
        if hasattr(value, "shape"):
            print(key, value.shape)
        else:
            print(key, type(value))

Ground truth columns: ['start', 'end', 'audiofilename', 'simultaneous_labels', 'LEPFRA', 'ELAPER', 'ENGPUS', 'DENEBR', 'DENMIC']
   start  end                         audiofilename  simultaneous_labels  \
0    0.0  1.0  Colombia/H23/H23_20230423_183500.WAV                    1   
1    1.0  2.0  Colombia/H23/H23_20230423_183500.WAV                    1   
2    2.0  3.0  Colombia/H23/H23_20230423_183500.WAV                    1   
3    3.0  4.0  Colombia/H23/H23_20230423_183500.WAV                    1   
4    4.0  5.0  Colombia/H23/H23_20230423_183500.WAV                    1   

   LEPFRA  ELAPER  ENGPUS  DENEBR  DENMIC  
0       0       0       0       0       1  
1       0       0       0       0       1  
2       0       0       0       0       1  
3       0       0       0       0       1  
4       0       0       0       0       1  


In [ ]:
# Train and evaluate the linear probe on marsh
probe, label2idx, metrics = bacpipe.probing_pipeline(
    model_name=MODEL_NAME,
    audio_dir=str(MARSH_DIR),
    ground_truth=marsh_gt,
    embeds=marsh_embeds,
    name="linear",
    overwrite=True,
)

print(metrics)
print(label2idx)


Found file: bacpipe_results/audio/evaluations/aves_especies/labels/probing_dataframe.csv. Loading dataframe probing dataframe. If you would like to automatically create a new probing dataframe. Please delete the existing one.

getting time of day: 100%|██████████| 104/104 [00:00<00:00, 355797.40it/s]
getting time per embeddings: 104it [00:00, 6891.12it/s]
getting day of year: 100%|██████████| 104/104 [00:00<00:00, 511380.56it/s]
getting continuous timestamps: 104it [00:00, 6981.00it/s]
getting parent directory: 104it [00:00, 24295.85it/s]
getting audio file names: 104it [00:00, 33822.41it/s]
Building metadata labels: 100%|██████████| 6/6 [00:00<00:00, 84.15it/s]

You have passed a multi-label ground truth array. However for visualization only one label will be displayed.

/workspaces/OsaAnurans3.11/bacpipe/embedding_evaluation/visualization/visualize_embeddings.py:465: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

{'overall': {'macro_accuracy': 0.7512494226631375, 'auc': 0.990052364310176, 'macro_f1': 0.7816726256965738}, 'items_per_class': {'DENMIC': 127, 'DENEBR': 30, 'ENGPUS': 159, 'ELAPER': 10, 'LEPFRA': 392}, 'per_class_accuracy': {'DENMIC': 0.952755905511811, 'DENEBR': 0.4, 'ENGPUS': 0.8238993710691824, 'ELAPER': 0.6, 'LEPFRA': 0.9795918367346939}, 'config': {'main_results_dir': 'bacpipe_results', 'embed_parent_dir': 'embeddings', 'dim_reduc_parent_dir': 'dim_reduced_embeddings', 'evaluations_dir': 'evaluations', 'model_base_path': 'bacpipe_model_checkpoints', 'global_batch_size': 8, 'audio_suffixes': ['.wav', '.WAV', '.aif', '.mp3', '.MP3', '.flac', '.ogg'], 'padding': 'wrap', 'avoid_pipelined_gpu_inference': False, 'nr_parallel_workers': False, 'rm_embedding_on_keyboard_interrupt': False, 'check_if_already_processed': True, 'check_if_already_dim_reduced': True, 'annotations_filename': 'annotations.csv', 'only_embed_annotations': False, 'min_annotation_length': 0, 'metadata_label_keys': [

In [43]:
# Load the trained probe and its species-to-index mapping
probe, label2index = bacpipe.prepare_probe_inference(
    model=MODEL_NAME,
    audio_dir=str(MARSH_DIR),
    main_results_dir=RESULTS_DIR,
)

# Get class probabilities for each swamp embedding
probabilities = bacpipe.run_probe_inference(
    model=MODEL_NAME,
    linear_probe=probe,
    embeds=swamp_embeds,
    audio_dir=str(MARSH_DIR),
    main_results_dir=RESULTS_DIR,
    return_binary_presence=False,
)

# Bacpipe's class mapping maps species names to probability columns.
index_to_label = {
    index: label for label, index in label2index.items()
}

swamp_species = np.array([
    index_to_label[i]
    for i in probabilities.argmax(axis=1)
])

swamp_confidence = probabilities.max(axis=1)

if len(swamp_embeds) != len(swamp_species):
    raise ValueError("Swamp embeddings and predictions are misaligned.")

In [44]:
# ---------------------------------------------------------------------
# 4. Fit UMAP on marsh, then project swamp into the same space
# ---------------------------------------------------------------------

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42,
)

marsh_2d = reducer.fit_transform(marsh_embeds)
swamp_2d = reducer.transform(swamp_embeds)

/workspaces/OsaAnurans3.11/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [46]:
marsh_2d

array([[22.266031, 11.291568],
       [22.14485 , 11.409028],
       [22.177748, 11.376518],
       ...,
       [16.83016 ,  9.164522],
       [16.734608,  9.275734],
       [16.815853,  9.146224]], dtype=float32)

In [47]:
import numpy as np

species_cols = ["LEPFRA", "ELAPER", "ENGPUS", "DENEBR", "DENMIC"]

# Ground-truth species for marsh.
# This chooses the first present species if multiple are present.
marsh_species = marsh_gt[species_cols].idxmax(axis=1).to_numpy()

# Mark intervals with no species label as unknown.
marsh_species[marsh_gt[species_cols].sum(axis=1).to_numpy() == 0] = "Unknown"

# Predicted species for swamp.
index_to_label = {index: label for label, index in label2index.items()}
swamp_species = np.array([
    index_to_label[i] for i in probabilities.argmax(axis=1)
])

# Confidence of the top predicted species.
swamp_confidence = probabilities.max(axis=1)

assert len(marsh_embeds) == len(marsh_species)
assert len(swamp_embeds) == len(swamp_species)

In [48]:
# ---------------------------------------------------------------------
# 5. Plot both datasets together
# ---------------------------------------------------------------------

plot_df = pd.concat(
    [
        pd.DataFrame({
            "UMAP 1": marsh_2d[:, 0],
            "UMAP 2": marsh_2d[:, 1],
            "dataset": "Marsh (known species)",
            "species": marsh_species,
            "confidence": np.nan,
        }),
        pd.DataFrame({
            "UMAP 1": swamp_2d[:, 0],
            "UMAP 2": swamp_2d[:, 1],
            "dataset": "Swamp (predicted species)",
            "species": swamp_species,
            "confidence": swamp_confidence,
        }),
    ],
    ignore_index=True,
)

# Save predictions for later inspection.
plot_df.to_csv("marsh_swamp_umap_points.csv", index=False)

fig, ax = plt.subplots(figsize=(13, 9))

species_names = sorted(plot_df["species"].unique())
palette = plt.get_cmap("tab20", max(len(species_names), 1))
species_colors = {
    name: palette(i)
    for i, name in enumerate(species_names)
}

# Plot marsh first, then swamp on top so predictions remain visible.
for species in species_names:
    marsh_mask = (
        (plot_df["dataset"] == "Marsh (known species)")
        & (plot_df["species"] == species)
    )
    swamp_mask = (
        (plot_df["dataset"] == "Swamp (predicted species)")
        & (plot_df["species"] == species)
    )

    if marsh_mask.any():
        ax.scatter(
            plot_df.loc[marsh_mask, "UMAP 1"],
            plot_df.loc[marsh_mask, "UMAP 2"],
            color=species_colors[species],
            marker="o",
            s=18,
            alpha=0.45,
        )

    if swamp_mask.any():
        ax.scatter(
            plot_df.loc[swamp_mask, "UMAP 1"],
            plot_df.loc[swamp_mask, "UMAP 2"],
            color=species_colors[species],
            marker="^",
            s=35,
            alpha=0.9,
            edgecolors="black",
            linewidths=0.25,
        )

# Build a legend with species colours.
from matplotlib.lines import Line2D

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        color=species_colors[name],
        label=name,
        markersize=7,
    )
    for name in species_names
]
legend_handles.extend([
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        color="grey",
        label="Marsh: known label",
        markersize=7,
    ),
    Line2D(
        [0], [0],
        marker="^",
        linestyle="",
        color="grey",
        markeredgecolor="black",
        label="Swamp: predicted label",
        markersize=7,
    ),
])

ax.set_title("Marsh vs Swamp — shared UMAP embedding space")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.legend(
    handles=legend_handles,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize="small",
)
fig.tight_layout()
fig.savefig("marsh_swamp_umap.png", dpi=200, bbox_inches="tight")
plt.show()

/tmp/ipykernel_29304/1741329761.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  plot_df = pd.concat(
